# 01 — Timestamp Correction (Aeris Ultra321 & Pico017)

The Aeris Ultra321 and Pico017 internal clocks may be misconfigured relative to UTC.
The RPi (WYO deployments) and Toughbook (MML deployments) logger files contain **both** the correct
UTC epoch and the instrument's timestamp in the same row, making the offset directly computable.

**Key insight**: The offset is NOT globally constant across the campaign — it varies by deployment
period (different clock states, instrument resets). We compute the offset per logger file in Part 2,
then in Part 3 each Aeris file is matched to the logger entry whose UTC window contains that file's
corrected first timestamp.

**Pipeline position:** `raw/` → **`ts_corrected/`** → `cleaned/` → `recleaned/` → `merged/`

## Workflow
- **Part 1**: Verify the offset on a substantial file pair (Ultra321 Feb 3, RPi)
- **Part 2**: Survey offset consistency across all logger files — confirms per-deployment variation
- **Part 3**: Match each Aeris file to its logger entry and apply its specific offset

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
import json
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from pathlib import Path

sys.path.insert(0, str(Path('../src').resolve()))
from timestamp_correction import (
    load_logger_file, compute_offset,
    apply_offset_to_raw, apply_offset_to_spectra,
    summarize_logger_files, build_coverage_map,
    batch_assign_offsets, AERIS_TS_FORMAT
)

RAW_DIR     = Path('/uufs/chpc.utah.edu/common/home/lin-group24/agm/Mobile_SLV/Data/2026/raw')
OUT_DIR     = Path('/uufs/chpc.utah.edu/common/home/lin-group24/agm/Mobile_SLV/Data/2026/ts_corrected')
OFFSETS_DIR = Path('../offsets')
OFFSETS_DIR.mkdir(exist_ok=True)

---
## Part 1 — Single-file verification (Ultra321, Feb 3 WYO)

We use the main RPi logger file from Feb 3 (`ultra_20260203_162046.dat`, ~26k records, 16:20–23:50 UTC)
and the first substantial Aeris raw file (`Ultra100321_260203_155038.txt`, ~7k records).

Note: `ultra_20260203_161119.dat` (48 lines) and `Ultra100321_260203_174513.txt` (208 lines) are
startup artifacts — small files created when the instrument restarts before the main log begins.

In [ ]:
# Load the main RPi Ultra logger file from Feb 3
rpi_path = RAW_DIR / 'LANL_rpi/Ultra/ultra_20260203_162046.dat'
rpi_df   = load_logger_file(rpi_path)

t_utc    = pd.to_datetime(rpi_df['Epoch_time'], unit='s', utc=True)
offset_s = compute_offset(rpi_df)

print(f'File          : {rpi_path.name}')
print(f'Records       : {len(rpi_df):,}')
print(f'UTC range     : {t_utc.iloc[0]}  ->  {t_utc.iloc[-1]}')
print(f'Offset median : {offset_s:.3f} s  ({offset_s/3600:.4f} hrs)')
print(f'Offset std    : {rpi_df["offset_s"].std():.4f} s')
rpi_df[['Epoch_time', 'Time Stamp', 'offset_s', 'CH4 (ppm)']].head(5)

In [ ]:
# Plot 1: offset stability + CH4 over the full deployment
# A flat offset line confirms no intra-file clock drift.
fig = make_subplots(
    rows=2, cols=1, shared_xaxes=True,
    subplot_titles=[f'Clock offset: Epoch_time - epoch(Time Stamp)  |  {rpi_path.name}', 'CH4 (ppm)'],
    vertical_spacing=0.08
)

fig.add_trace(go.Scatter(
    x=t_utc, y=rpi_df['offset_s'], mode='lines',
    line=dict(width=0.8, color='steelblue'), name='offset (s)', showlegend=False
), row=1, col=1)

fig.add_hline(
    y=offset_s, line_dash='dash', line_color='red', line_width=1.5,
    annotation_text=f'median = {offset_s:.1f} s ({offset_s/3600:.4f} hrs)',
    annotation_position='top right', row=1, col=1
)

fig.add_trace(go.Scatter(
    x=t_utc, y=rpi_df['CH4 (ppm)'], mode='lines',
    line=dict(width=0.8, color='black'), name='CH4', showlegend=False
), row=2, col=1)

fig.update_yaxes(title_text='Offset (s)', row=1, col=1)
fig.update_yaxes(title_text='CH4 (ppm)', row=2, col=1)
fig.update_xaxes(title_text='Time (UTC)', row=2, col=1)
fig.update_layout(height=500)
fig.show()

In [ ]:
# Load the first substantial Aeris Ultra321 raw file from Feb 3
aeris_path = RAW_DIR / 'LANL_aerisultra321/Raw/Ultra100321_260203_155038.txt'
aeris_df   = pd.read_csv(aeris_path)
aeris_df.columns = aeris_df.columns.str.strip()

ts_wrong     = pd.to_datetime(aeris_df['Time Stamp'].str.strip(), format=AERIS_TS_FORMAT)
ts_corrected = (ts_wrong + pd.Timedelta(seconds=offset_s)).dt.tz_localize('UTC')

print(f'Aeris file    : {aeris_path.name}')
print(f'Records       : {len(aeris_df):,}')
print(f'Wrong range   : {ts_wrong.iloc[0]}  ->  {ts_wrong.iloc[-1]}')
print(f'Corrected UTC : {ts_corrected.iloc[0]}  ->  {ts_corrected.iloc[-1]}')

In [ ]:
# Plot 2: overlay RPi CH4 (correct UTC) and Aeris CH4 (corrected timestamps).
# The two traces should overlap in the window where both files have data.
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=t_utc, y=rpi_df['CH4 (ppm)'], mode='lines',
    line=dict(width=2), opacity=0.7,
    name=f'RPi logger — correct UTC ({rpi_path.name})'
))

fig.add_trace(go.Scatter(
    x=ts_corrected, y=aeris_df['CH4 (ppm)'], mode='lines',
    line=dict(width=1.5, dash='dash', color='tomato'), opacity=0.9,
    name=f'Aeris raw — corrected +{offset_s:.1f}s ({aeris_path.name})'
))

fig.update_layout(
    title='Verification: RPi (correct epoch) vs Aeris (corrected timestamps) — should overlap where both have data',
    xaxis_title='Time (UTC)',
    yaxis_title='CH4 (ppm)',
    height=400,
    legend=dict(yanchor='top', y=0.99, xanchor='left', x=0.01)
)
fig.show()

---
## Part 2 — Offset consistency across all logger files

Compute the offset from every RPi and Toughbook logger file. If the offset is consistent
(low std, similar median across all files), a single global value per instrument is appropriate.
Outlier files should be inspected before proceeding to Part 3.

In [ ]:
print('=== LANL_rpi / Ultra ===')
rpi_ultra_summary = summarize_logger_files(RAW_DIR / 'LANL_rpi/Ultra')
pd.set_option('display.max_colwidth', 45)
rpi_ultra_summary

In [ ]:
print('=== LANL_rpi / Pico ===')
rpi_pico_summary = summarize_logger_files(RAW_DIR / 'LANL_rpi/Pico')
rpi_pico_summary

In [ ]:
print('=== LANL_toughbook / Ultra ===')
tb_ultra_summary = summarize_logger_files(RAW_DIR / 'LANL_toughbook/Ultra')
tb_ultra_summary

In [ ]:
print('=== LANL_toughbook / Pico ===')
tb_pico_summary = summarize_logger_files(RAW_DIR / 'LANL_toughbook/Pico')
tb_pico_summary

In [ ]:
# Pool and plot offset consistency — outliers stand out immediately
all_ultra = pd.concat([rpi_ultra_summary, tb_ultra_summary], ignore_index=True)
all_pico  = pd.concat([rpi_pico_summary,  tb_pico_summary],  ignore_index=True)

fig = make_subplots(
    rows=2, cols=1,
    subplot_titles=['Ultra321 (RPi + Toughbook)', 'Pico017 (RPi + Toughbook)'],
    vertical_spacing=0.15
)

for row, (summary, label) in enumerate([
    (all_ultra, 'Ultra321'),
    (all_pico,  'Pico017')
], start=1):
    s   = summary.dropna(subset=['offset_median_s'])
    med = float(s['offset_median_s'].median())

    fig.add_trace(go.Scatter(
        x=s['filename'], y=s['offset_median_s'],
        error_y=dict(type='data', array=s['offset_std_s'] * 3, visible=True),
        mode='markers', marker=dict(size=8),
        name=label, showlegend=False
    ), row=row, col=1)

    fig.add_hline(
        y=med, line_dash='dash', line_color='red', line_width=1,
        annotation_text=f'global median = {med:.1f} s',
        annotation_position='top right', row=row, col=1
    )
    fig.update_yaxes(title_text='Offset (s)', row=row, col=1)

fig.update_xaxes(tickangle=45, row=1, col=1)
fig.update_xaxes(tickangle=45, row=2, col=1)
fig.update_layout(height=700)
fig.show()

In [ ]:
# Summary: global medians for reference — note the large spread confirms per-deployment matching is needed.
ultra_global_median = float(all_ultra['offset_median_s'].dropna().median())
pico_global_median  = float(all_pico ['offset_median_s'].dropna().median())

print(f'Global median — Ultra321 : {ultra_global_median:.1f} s  (spread: {all_ultra["offset_median_s"].dropna().std():.1f} s std)')
print(f'Global median — Pico017  : {pico_global_median:.1f} s  (spread: {all_pico["offset_median_s"].dropna().std():.1f} s std)')
print()
print('A single global offset would be wrong by up to tens of seconds for some files.')
print('Part 3 assigns each Aeris file its own offset via per-deployment matching.')

---
## Part 3 — Batch: apply per-deployment offset to all Raw + Spectra files

Offsets vary across the campaign (Part 2 confirmed this). Each Aeris file is matched to the logger
entry whose corrected UTC window contains that file's first timestamp. `batch_assign_offsets` tries
each logger entry's `offset_median_s` and checks whether the corrected timestamp falls within that
logger's UTC window (±2 h buffer).

**Step 1** — Preview all assignments: check for any unmatched files (`status != 'ok'`).  
**Step 2** — Plot per-file assigned offsets: visually confirm assignments look sensible.  
**Step 3** — Apply and write to `ts_corrected/`. **Only run after reviewing Steps 1 and 2.**

Output mirrors the `raw/` directory structure under `ts_corrected/`:

    ts_corrected/
      LANL_aerisultra321/Raw/
      LANL_aerisultra321/Spectra/
      LANL_aerispico017/Raw/
      LANL_aerispico017/Spectra/

In [ ]:
# Build per-instrument coverage maps from Part 2 summaries.
# Files with < 200 records (startup artifacts) and errored files are excluded automatically.
# Duplicate filenames across RPi and Toughbook dirs are deduplicated (first occurrence wins).
ultra_coverage = build_coverage_map(rpi_ultra_summary, tb_ultra_summary)
pico_coverage  = build_coverage_map(rpi_pico_summary,  tb_pico_summary)

print(f'Ultra coverage entries : {len(ultra_coverage)}')
for e in ultra_coverage:
    print(f"  [{e['start_utc']}  ->  {e['end_utc']}]  offset={e['offset_s']:+.1f}s  ({e['logger_filename']})")

print()
print(f'Pico  coverage entries : {len(pico_coverage)}')
for e in pico_coverage:
    print(f"  [{e['start_utc']}  ->  {e['end_utc']}]  offset={e['offset_s']:+.1f}s  ({e['logger_filename']})")

In [ ]:
# Assign offsets to all Aeris files — preview only, nothing written yet.
ultra_raw_files     = sorted((RAW_DIR / 'LANL_aerisultra321/Raw').glob('*.txt'))
ultra_spectra_files = sorted((RAW_DIR / 'LANL_aerisultra321/Spectra').glob('*.txt'))
pico_raw_files      = sorted((RAW_DIR / 'LANL_aerispico017/Raw').glob('*.txt'))
pico_spectra_files  = sorted((RAW_DIR / 'LANL_aerispico017/Spectra').glob('*.txt'))

ultra_raw_assign     = batch_assign_offsets(ultra_raw_files,     ultra_coverage, file_type='raw')
ultra_spectra_assign = batch_assign_offsets(ultra_spectra_files, ultra_coverage, file_type='spectra')
pico_raw_assign      = batch_assign_offsets(pico_raw_files,      pico_coverage,  file_type='raw')
pico_spectra_assign  = batch_assign_offsets(pico_spectra_files,  pico_coverage,  file_type='spectra')

for label, df in [
    ('Ultra321 Raw',     ultra_raw_assign),
    ('Ultra321 Spectra', ultra_spectra_assign),
    ('Pico017 Raw',      pico_raw_assign),
    ('Pico017 Spectra',  pico_spectra_assign),
]:
    n_ok   = (df['status'] == 'ok').sum()
    n_fail = (df['status'] != 'ok').sum()
    status = 'ALL MATCHED' if n_fail == 0 else f'{n_fail} UNMATCHED — investigate before applying'
    print(f'{label:20s}: {n_ok:3d} matched  |  {status}')
    if n_fail:
        print(df[df['status'] != 'ok'][['filename', 'status']].to_string(index=False))

In [ ]:
# Plot per-file assigned offsets — each dot is one Aeris file, hover shows which logger matched it.
# Files within the same deployment period should cluster at the same offset value.
fig = make_subplots(
    rows=2, cols=1,
    subplot_titles=['Ultra321 — assigned offset per Raw file', 'Pico017 — assigned offset per Raw file'],
    vertical_spacing=0.15
)

for row, (assign_df, label) in enumerate([
    (ultra_raw_assign, 'Ultra321'),
    (pico_raw_assign,  'Pico017'),
], start=1):
    ok = assign_df[assign_df['status'] == 'ok']
    fig.add_trace(go.Scatter(
        x=ok['filename'], y=ok['offset_s'],
        mode='markers', marker=dict(size=8),
        text=ok['logger_filename'],
        hovertemplate='%{x}<br>offset=%{y:.1f} s<br>logger=%{text}<extra></extra>',
        name=label, showlegend=False
    ), row=row, col=1)
    fig.update_yaxes(title_text='Offset (s)', row=row, col=1)

fig.update_xaxes(tickangle=45, row=1, col=1)
fig.update_xaxes(tickangle=45, row=2, col=1)
fig.update_layout(height=700)
fig.show()

In [ ]:
# Apply Ultra321 Raw — per-file offset
print(f'Ultra321 Raw: {len(ultra_raw_files)} files\n')
for _, row in ultra_raw_assign.iterrows():
    if row['status'] != 'ok':
        print(f'  SKIP  {row["filename"]}  ({row["status"]})')
        continue
    in_path  = RAW_DIR / 'LANL_aerisultra321/Raw' / row['filename']
    out_path = OUT_DIR / 'LANL_aerisultra321/Raw' / row['filename']
    apply_offset_to_raw(in_path, row['offset_s'], out_path)
    print(f'  OK  {row["offset_s"]:+.1f}s  {row["filename"]}')
print('\nDone.')

In [ ]:
# Apply Ultra321 Spectra (large files — may take several minutes total)
print(f'Ultra321 Spectra: {len(ultra_spectra_files)} files\n')
for _, row in ultra_spectra_assign.iterrows():
    if row['status'] != 'ok':
        print(f'  SKIP  {row["filename"]}  ({row["status"]})')
        continue
    in_path  = RAW_DIR / 'LANL_aerisultra321/Spectra' / row['filename']
    out_path = OUT_DIR / 'LANL_aerisultra321/Spectra' / row['filename']
    apply_offset_to_spectra(in_path, row['offset_s'], out_path)
    print(f'  OK  {row["offset_s"]:+.1f}s  {row["filename"]}')
print('\nDone.')

In [ ]:
# Apply Pico017 Raw — per-file offset
print(f'Pico017 Raw: {len(pico_raw_files)} files\n')
for _, row in pico_raw_assign.iterrows():
    if row['status'] != 'ok':
        print(f'  SKIP  {row["filename"]}  ({row["status"]})')
        continue
    in_path  = RAW_DIR / 'LANL_aerispico017/Raw' / row['filename']
    out_path = OUT_DIR / 'LANL_aerispico017/Raw' / row['filename']
    apply_offset_to_raw(in_path, row['offset_s'], out_path)
    print(f'  OK  {row["offset_s"]:+.1f}s  {row["filename"]}')
print('\nDone.')

In [ ]:
# Apply Pico017 Spectra (large files — may take several minutes total)
print(f'Pico017 Spectra: {len(pico_spectra_files)} files\n')
for _, row in pico_spectra_assign.iterrows():
    if row['status'] != 'ok':
        print(f'  SKIP  {row["filename"]}  ({row["status"]})')
        continue
    in_path  = RAW_DIR / 'LANL_aerispico017/Spectra' / row['filename']
    out_path = OUT_DIR / 'LANL_aerispico017/Spectra' / row['filename']
    apply_offset_to_spectra(in_path, row['offset_s'], out_path)
    print(f'  OK  {row["offset_s"]:+.1f}s  {row["filename"]}')
print('\nDone.')

In [ ]:
# Save per-file offset assignments to mobile-slv/offsets/ (version controlled with code)
def _to_records(assign_df):
    return [
        {'filename': r['filename'], 'offset_s': r['offset_s'], 'logger': r['logger_filename']}
        for _, r in assign_df[assign_df['status'] == 'ok'].iterrows()
    ]

offset_record = {
    'ultra321_raw':     _to_records(ultra_raw_assign),
    'ultra321_spectra': _to_records(ultra_spectra_assign),
    'pico017_raw':      _to_records(pico_raw_assign),
    'pico017_spectra':  _to_records(pico_spectra_assign),
}
out_json = OFFSETS_DIR / 'ts_correction_offsets.json'
with open(out_json, 'w') as f:
    json.dump(offset_record, f, indent=2)

total = sum(len(v) for v in offset_record.values())
print(f'Saved: {out_json}  ({total} file assignments)')